In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml pypdf
dbutils.library.restartPython()


In [0]:
# =============================================================================
# Imports
# =============================================================================
import os
import re
import io
import json
import time
import random
import hashlib
import unicodedata
from datetime import datetime, timezone
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests
from pypdf import PdfReader


In [0]:
# Carrega a função compartilhada que atualiza o status real da fonte na
# tabela de controle a cada execução.
%run "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/scripts/utils_controle"


In [0]:
# =============================================================================
# Configuração
# =============================================================================

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")
HOJE_BR = datetime.now(timezone.utc).astimezone().strftime("%d/%m/%Y")

SITE_URL = "https://www.diariooficial.ms.gov.br/"
SOURCE_ID = "diario_oficial_ms"
SOURCE_DESCRICAO = "Linked from Diário Oficial de Mato Grosso do Sul"

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}/GERAL"
os.makedirs(PASTA_DESTINO, exist_ok=True)
print(f"[setup] Salvando artefatos em: {PASTA_DESTINO}")

# Manifesto — evita reprocessar a mesma edição se o job rodar mais de uma vez
# no mesmo dia, ou se a edição de hoje ainda não tiver sido publicada ontem.
PASTA_MANIFESTOS = "/Volumes/desafio_kinea/research/research_volume/infraestrutura/manifests"
os.makedirs(PASTA_MANIFESTOS, exist_ok=True)
CAMINHO_MANIFESTO = os.path.join(PASTA_MANIFESTOS, f"{SOURCE_ID}_processados.json")

# Termos que definem se uma PÁGINA do diário é relevante o bastante pra
# passar pro processamento seguinte. Ajuste conforme o que aparecer nos
# primeiros testes reais — isso é o primeiro rascunho, não validado ainda.
PALAVRAS_CHAVE_INFRA = [
    "AGEMS", "saneamento", "energia elétrica", "tarifa", "concessão",
    "concessionária", "transporte rodoviário", "regulação", "regulatório",
    "distribuição de energia", "abastecimento de água", "esgotamento sanitário",
]

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
]

IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

HTTP_TIMEOUT = 60  # PDFs de diário podem ser grandes; timeout maior que o usual


In [0]:
# =============================================================================
# Helpers
# =============================================================================

def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def _normalizar(s: str) -> str:
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii").upper()


_PALAVRAS_CHAVE_NORM = [_normalizar(p) for p in PALAVRAS_CHAVE_INFRA]


def pagina_e_relevante(texto_pagina: str) -> list[str]:
    """Retorna a lista de palavras-chave encontradas na página (vazia se nenhuma)."""
    texto_norm = _normalizar(texto_pagina)
    return [p for p, p_norm in zip(PALAVRAS_CHAVE_INFRA, _PALAVRAS_CHAVE_NORM) if p_norm in texto_norm]


def carregar_manifesto(caminho: str) -> set:
    if not os.path.exists(caminho):
        return set()
    try:
        with open(caminho, "r", encoding="utf-8") as f:
            return set(json.load(f))
    except Exception as e:
        print(f"[manifesto] falha ao carregar ({e}); iniciando vazio.")
        return set()


def salvar_manifesto(caminho: str, urls: set) -> None:
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(sorted(urls), f, ensure_ascii=False, indent=2)


def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",  # sem "br" — evita o bug de Brotli sem decoder
    }
    if referer:
        headers["Referer"] = referer
    return headers


In [0]:
# =============================================================================
# Etapa 1 — Encontrar a edição de hoje na home e extrair a URL do PDF
# =============================================================================

def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer="https://www.diariooficial.ms.gov.br/")
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None


def encontrar_edicao_do_dia(html: str, data_alvo_br: str) -> Optional[dict]:
    """
    Procura, na tabela "Últimas Edições Publicadas", a linha cuja data bate
    com data_alvo_br (formato DD/MM/AAAA). Retorna {"titulo", "url"} ou None
    se a edição de hoje ainda não tiver sido publicada (comum se o job rodar
    muito cedo, antes da publicação diária).
    """
    soup = BeautifulSoup(html, "lxml")

    for tag_a in soup.find_all("a", href=True):
        href = tag_a["href"].strip()
        if not href.lower().endswith(".pdf"):
            continue

        # A data costuma aparecer no texto da linha/célula próxima ao link;
        # como fallback mais robusto, procura o padrão DD_MM_AAAA no próprio
        # nome do arquivo, que é como a URL é montada.
        m = re.search(r"(\d{2})_(\d{2})_(\d{4})\.pdf$", href)
        if not m:
            continue
        dia, mes, ano = m.groups()
        data_do_link = f"{dia}/{mes}/{ano}"

        if data_do_link == data_alvo_br:
            titulo = tag_a.get_text(" ", strip=True) or f"Diário Oficial {data_do_link}"
            return {"titulo": titulo, "url": href}

    return None


In [0]:
# =============================================================================
# Etapa 2 — Baixar o PDF e filtrar páginas relevantes
# =============================================================================

def baixar_pdf(url: str, tentativas: int = 3) -> Optional[bytes]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            content_type = resp.headers.get("content-type", "")
            if resp.status_code == 200 and "pdf" in content_type.lower() and resp.content:
                return resp.content
            print(f"  [pdf tent {tentativa}/{tentativas}] status={resp.status_code} "
                  f"content-type={content_type!r}")
        except Exception as e:
            print(f"  [pdf tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.5, 3.0))

    return None


def extrair_paginas_relevantes(conteudo_pdf: bytes) -> list[dict]:
    """
    Percorre o PDF página por página. Só mantém as páginas cujo texto bate
    com algum termo de PALAVRAS_CHAVE_INFRA — evita mandar o diário inteiro
    (que pode ter centenas de páginas) pro processamento seguinte.
    """
    reader = PdfReader(io.BytesIO(conteudo_pdf))
    total_paginas = len(reader.pages)
    print(f"  PDF com {total_paginas} páginas — filtrando por palavra-chave...")

    paginas_relevantes = []
    for i, pagina in enumerate(reader.pages, start=1):
        texto_pagina = pagina.extract_text() or ""
        termos_encontrados = pagina_e_relevante(texto_pagina)
        if termos_encontrados:
            paginas_relevantes.append({
                "pagina": i,
                "termos": termos_encontrados,
                "texto": texto_pagina.strip(),
            })

    print(f"  {len(paginas_relevantes)} de {total_paginas} páginas mencionam termos-alvo.")
    return paginas_relevantes


In [0]:
# =============================================================================
# Etapa 3 — Salvar no Volume
# =============================================================================

def salvar_artefatos(pasta: str, titulo: str, paginas: list[dict], metadados: dict) -> tuple[str, str]:
    slug_source = slugify(SOURCE_ID, max_len=40)
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    # Texto final: cada página relevante, com cabeçalho indicando o nº da
    # página real no PDF original e os termos que a fizeram ser selecionada
    # — facilita auditoria/conferência manual depois.
    blocos = []
    for p in paginas:
        cabecalho = f"--- Página {p['pagina']} (termos: {', '.join(p['termos'])}) ---"
        blocos.append(f"{cabecalho}\n{p['texto']}")
    texto_final = "\n\n".join(blocos)

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto_final)

    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json


In [0]:
# =============================================================================
# Execução
# =============================================================================

try:
    html_home = baixar_pagina(SITE_URL)
    if not html_home:
        raise RuntimeError("Não foi possível baixar a home do Diário Oficial.")

    edicao = encontrar_edicao_do_dia(html_home, HOJE_BR)

    if not edicao:
        print(f"=== Edição de hoje ({HOJE_BR}) ainda não encontrada na listagem. "
              f"Pode não ter sido publicada ainda, ou o job rodou antes da hora. ===")
        atualizar_status_fonte(source_id=SOURCE_ID, sucesso=True, docs_capturados=0)
    else:
        ja_processados = carregar_manifesto(CAMINHO_MANIFESTO)

        if edicao["url"] in ja_processados:
            print(f"=== Edição de hoje já processada anteriormente; nada a fazer. ===")
            atualizar_status_fonte(source_id=SOURCE_ID, sucesso=True, docs_capturados=0)
        else:
            print(f"=== Edição encontrada: {edicao['titulo']!r} ===")

            conteudo_pdf = baixar_pdf(edicao["url"])
            if not conteudo_pdf:
                print("-> download do PDF falhou.")
                atualizar_status_fonte(
                    source_id=SOURCE_ID, sucesso=False, docs_capturados=0,
                    erro="download do PDF da edicao falhou",
                )
            else:
                paginas_relevantes = extrair_paginas_relevantes(conteudo_pdf)

                if not paginas_relevantes:
                    print("-> nenhuma página relevante encontrada nesta edição.")
                    ja_processados.add(edicao["url"])  # evita reprocessar; não achou nada mesmo
                    atualizar_status_fonte(source_id=SOURCE_ID, sucesso=True, docs_capturados=0)
                else:
                    metadados = {
                        "source_id": SOURCE_ID,
                        "title": edicao["titulo"],
                        "description": SOURCE_DESCRICAO,
                        "url": edicao["url"],
                        "date": HOJE,
                        "published_at": HOJE,
                        "qtd_paginas_relevantes": len(paginas_relevantes),
                    }

                    caminho_txt, caminho_json = salvar_artefatos(
                        PASTA_DESTINO, edicao["titulo"], paginas_relevantes, metadados
                    )
                    print(f"-> salvo em {caminho_txt}")
                    ja_processados.add(edicao["url"])
                    atualizar_status_fonte(source_id=SOURCE_ID, sucesso=True, docs_capturados=1)

            salvar_manifesto(CAMINHO_MANIFESTO, ja_processados)

except Exception as e:
    print(f"=== ERRO GERAL: {e} ===")
    atualizar_status_fonte(source_id=SOURCE_ID, sucesso=False, docs_capturados=0, erro=str(e))

print("\n=== Fim. ===")
